In [1]:
%load_ext autoreload
%autoreload 2

import pickle
import ujson
import json
import sys
import os
from collections import defaultdict

import pandas as pd
import numpy as np
import torch
import random
import faiss

from tqdm import tqdm
from collections import defaultdict
from typing import Optional

from bioel.utils.umls_utils import UmlsMappings
from bioel.utils.bigbio_utils import CUIS_TO_REMAP, CUIS_TO_EXCLUDE, DATASET_NAMES, VALIDATION_DOCUMENT_IDS
from bioel.utils.bigbio_utils import load_bigbio_dataset, add_deabbreviations, load_dataset_df, dataset_to_documents, dataset_to_df, load_dataset_df, resolve_abbreviation, dataset_unique_tax_ids
from bioel.utils.solve_abbreviation.solve_abbreviations import create_abbrev

from bioel.ontology import BiomedicalOntology
from bioel.models.arboel.biencoder.data.data_utils import process_ontology
from bioel.evaluate import Evaluate

from torch.utils.data import DataLoader

from peft import PeftModel
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from ids import open_ai_api_key

/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING 09-28 20:57:55 cuda.py:22] You are using a deprecated `pynvml` package. Please install `nvidia-ml-py` instead, and make sure to uninstall `pynvml`. When both of them are installed, `pynvml` will take precedence and cause errors. See https://pypi.org/project/pynvml for more information.
WARNING 09-28 20:57:55 cuda.py:69] Detected different devices in the system: 
WARNING 09-28 20:57:55 cuda.py:69] Tesla V100-PCIE-32GB
WARNING 09-28 20:57:55 cuda.py:69] NVIDIA A40
WARNING 09-28 20:57:55 cuda.py:69] NVIDIA A40
WARNING 09-28 20:57:55 cuda.py:69] NVIDIA A40
WARNING 09-28 20:57:55 cuda.py:69] Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.


2024-09-28 20:57:55,912	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
from utils_functions import *

In [3]:
import openai
openai.api_key = open_ai_api_key
import re
import ujson
import logging
from collections import Counter, defaultdict
import pandas as pd

# Set up logging configuration
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logging.getLogger("httpx").setLevel(logging.WARNING)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1" 
sampling_params = SamplingParams(temperature=0, top_p=0.9, max_tokens=1000, stop=["<|eot_id|>"])

In [4]:
# Check how many GPUs are visible
print(f"Visible GPUs: {torch.cuda.device_count()}")

# Check if you can access both GPUs
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    
device_1 = torch.device("cuda:0")  # First GPU (GPU 0)
device_2 = torch.device("cuda:1")  # Second GPU (GPU 1)

seed = 40
np.random.seed(seed)
random.seed(seed)

Visible GPUs: 2
GPU 0: NVIDIA A40
GPU 1: NVIDIA A40


In [40]:
# dataset_name = 'ncbi_disease'

# ontology_dir = "/mitchell/entity-linking/kbs/medic.tsv"
# name = "medic"
# ontology2 = BiomedicalOntology.load_medic(filepath=ontology_dir, name=name)

# dataset_name = "gnormplus"
# dataset_name = "nlm_gene"

# entrez_dict = {"name" : "entrez",
#              "filepath" : "/mitchell/entity-linking/el-robustness-comparison/data/gene_info.tsv",
#              "dataset" : f"{dataset_name}",}
# ontology2 = BiomedicalOntology.load_entrez(**entrez_dict)

# dataset_name = "nlmchem"
# mesh_dict = {"name" : "mesh",
#              "filepath" : "/mitchell/entity-linking/2017AA/META/"}
# ontology2 = BiomedicalOntology.load_mesh(**mesh_dict)

dataset_name = "medmentions_st21pv"
umls_dict_st21pv = {
    "name": "umls",
    "filepath": "/mitchell/entity-linking/2017AA/META/",
    "path_st21pv_cui": "/home2/cye73/data_test2/arboel/medmentions_st21pv/umls_cuis_st21pv.json",
}
ontology2 = BiomedicalOntology.load_umls(**umls_dict_st21pv)



EL_model = "sapbert"
# EL_model = "crossencoder"


[2024-09-28 21:46:44] [ontology.py] [INFO] Reading UMLS from /mitchell/entity-linking/2017AA/META/


Loading cached UMLS data from /mitchell/entity-linking/2017AA/META/.cached_df.feather


/home2/cye73/biomedical-entity-linking/bioel/bioel/utils/umls_utils.py:207: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["identifier"] = df["cui"]
/home2/cye73/biomedical-entity-linking/bioel/bioel/ontology.py:378: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda x: list(set(x[1]) - set([x[0]])), axis=1


Number of entities : 2369483


Loading UMLS Ontology: 2369483it [02:10, 18205.28it/s]


In [41]:
path_to_abbrev = "/home2/cye73/data_test2/abbreviations.json"
dataset = load_bigbio_dataset(dataset_name)
dataset = add_deabbreviations(dataset, path_to_abbrev)

[autoreload of bioel.evaluate failed: Traceback (most recent call last):
  File "/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
  File "/nethome/cye73/conda_envs/bioel/lib/python3.9/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 613, in _exec
  File "<frozen importlib._bootstrap_external>", line 850, in exec_module
  File "<frozen importlib._bootstrap>", line 228, in _call_with_frames_removed
  File "/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py", line 286, in <module>
    class Evaluate:
  File "/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py", line 617, in Evaluate
    def detailed_results(self, k=self.max_k, eval

In [42]:
dataset_df = dataset_to_df(dataset)
test_df = dataset_df[dataset_df['split'] == 'test']
train_df = dataset_df[dataset_df['split'] == 'train']
test_df

,document_id,offsets,text,type,db_ids,split,deabbreviated_text,mention_id
82,25847295,"[[34, 43]]",apoptosis,[T038],[UMLS:C0162638],test,apoptosis,25847295.1
87,25847295,"[[55, 65]]",PC12 cells,[T017],[UMLS:C0085262],test,PC12 cells,25847295.2
73,25847295,"[[137, 144]]",present,[T033],[UMLS:C0150312],test,present,25847295.3
78,25847295,"[[206, 219]]",toxic effects,[T037],[UMLS:C0600688],test,toxic effects,25847295.4
79,25847295,"[[259, 268]]",Apoptosis,[T038],[UMLS:C0162638],test,Apoptosis,25847295.5
...,...,...,...,...,...,...,...,...
203011,28550165,"[[1665, 1695]]",microtubule-associated protein,[T103],[UMLS:C0026045],test,microtubule-associated protein,28550165.90
203012,28550165,"[[1725, 1743]]",metabolic function,[T038],[UMLS:C0597299],test,metabolic function,28550165.91
203014,28550165,"[[1813, 1825]]",microtubules,[T017],[UMLS:C0026046],test,microtubules,28550165.92
203015,28550165,"[[1843, 1855]]",cytoskeleton,[T017],[UMLS:C0010853],test,cytoskeleton,28550165.93


In [43]:
docs = dataset_to_documents(dataset)
# docs

In [44]:
add_full_context(df = test_df, docs=docs)
add_full_context(df = train_df, docs=docs)
# test_df

/home2/cye73/llm_disambiguator/experiments/utils_functions.py:375: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["contextualized_mention"] = contextualized_mentions
/home2/cye73/llm_disambiguator/experiments/utils_functions.py:375: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["contextualized_mention"] = contextualized_mentions


In [45]:
_, TestMap_mention2context = add_context(df = test_df, docs = docs)
corpus, TrainMap_mention2context = add_context(df = train_df, docs = docs)
_, _ = add_context(df = dataset_df, docs = docs)
# dataset_df

/home2/cye73/llm_disambiguator/experiments/utils_functions.py:421: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["limited_contextualized_mention"] = limited_contextualized_mentions
/home2/cye73/llm_disambiguator/experiments/utils_functions.py:421: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["limited_contextualized_mention"] = limited_contextualized_mentions


In [46]:
# TrainMap_mention2context
# print(len(corpus))

In [47]:
# # # Load a pre-trained SentenceBERT model
# # model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# # # Generate embeddings for the corpus
# # corpus_embeddings = model.encode(corpus, convert_to_tensor=True)

# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer('princeton-nlp/sup-simcse-bert-base-uncased')
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# model.to(device_1)
# # Generate embeddings for the corpus
# corpus_embeddings = model.encode(corpus, convert_to_tensor=True)

# corpus_embeddings = corpus_embeddings.cpu().detach().numpy()

In [48]:
# embedding_dimension = corpus_embeddings.shape[1]

# # Create the HNSW index with the correct arguments
# M = 32  # Number of neighbors in the HNSW graph
# index = faiss.IndexHNSWFlat(embedding_dimension, M)

# # Normalize the corpus embeddings if using cosine similarity
# faiss.normalize_L2(corpus_embeddings)

# # Add the embeddings to the index
# index.add(corpus_embeddings)

# # Print the number of sentences added to the index
# print(f"Number of sentences in the index: {index.ntotal}")

In [49]:
TrainMap_context2mention = {v: k for k, v in TrainMap_mention2context.items()}
# TrainMap_context2mention

In [50]:
# with open(f"/home2/cye73/results2/sapbert/sapbert_{dataset_name}.json", "r") as f:
#     sapbert_res = ujson.load(f)
# sapbert_res

[{'document_id': '25847295',
  'offsets': [[34, 43]],
  'text': 'apoptosis',
  'type': ['T038'],
  'db_ids': ['UMLS:C0162638'],
  'split': 'test',
  'deabbreviated_text': 'apoptosis',
  'mention_id': '25847295.1.abbr_resolved',
  'candidates': [['UMLS:C0162638', 'UMLS:C0162638'],
   ['UMLS:C0162638'],
   ['UMLS:C0162638', 'UMLS:C0162638'],
   ['UMLS:C0162638'],
   ['UMLS:C0162638'],
   ['UMLS:C0162638'],
   ['UMLS:C1516044'],
   ['UMLS:C0162638'],
   ['UMLS:C1159821'],
   ['UMLS:C3269242']],
  'candidates_metadata': [{'text': 'apoptosis',
    'db_id': 'UMLS:C0162638|UMLS:C0162638'},
   {'text': 'apoptosis (morphologic abnormality)', 'db_id': 'UMLS:C0162638'},
   {'text': 'apoptotic process', 'db_id': 'UMLS:C0162638|UMLS:C0162638'},
   {'text': 'programmed cell death by apoptosis', 'db_id': 'UMLS:C0162638'},
   {'text': 'apoptotic programmed cell death', 'db_id': 'UMLS:C0162638'},
   {'text': 'apoptotic cell death', 'db_id': 'UMLS:C0162638'},
   {'text': 'apoptotic', 'db_id': 'UMLS:C151

In [51]:
# processed_res = process_candidates_sapbert(sapbert_res)
# processed_res

[{'document_id': '25847295',
  'offsets': [[34, 43]],
  'text': 'apoptosis',
  'type': ['T038'],
  'db_ids': ['UMLS:C0162638'],
  'split': 'test',
  'deabbreviated_text': 'apoptosis',
  'mention_id': '25847295.1.abbr_resolved',
  'candidates': [['UMLS:C0162638'],
   ['UMLS:C1516044'],
   ['UMLS:C1159821'],
   ['UMLS:C3269242'],
   ['ERROR_IGNORE_THIS'],
   ['ERROR_IGNORE_THIS'],
   ['ERROR_IGNORE_THIS'],
   ['ERROR_IGNORE_THIS'],
   ['ERROR_IGNORE_THIS'],
   ['ERROR_IGNORE_THIS']],
  'candidates_metadata': [{'text': 'apoptosis',
    'db_id': 'UMLS:C0162638|UMLS:C0162638'},
   {'text': 'apoptosis (morphologic abnormality)', 'db_id': 'UMLS:C0162638'},
   {'text': 'apoptotic process', 'db_id': 'UMLS:C0162638|UMLS:C0162638'},
   {'text': 'programmed cell death by apoptosis', 'db_id': 'UMLS:C0162638'},
   {'text': 'apoptotic programmed cell death', 'db_id': 'UMLS:C0162638'},
   {'text': 'apoptotic cell death', 'db_id': 'UMLS:C0162638'},
   {'text': 'apoptotic', 'db_id': 'UMLS:C1516044'},
  

In [53]:
# with open(f"/home2/cye73/results2/sapbert/sapbert_{dataset_name}_real.json", "w") as f:
#     ujson.dump(processed_res, f, indent=4)

# Load arboel results

In [55]:
dataset_names = [f"{dataset_name}"]

model_names = ["arboel_biencoder", "arboel_crossencoder", "sapbert"]
path_to_result = {f"{dataset_name}": {
        # "arboel_biencoder": f"/home2/cye73/results2/arboel/{dataset_name}/biencoder_output_eval.json",
        # "arboel_crossencoder": f"/home2/cye73/results2/arboel/{dataset_name}/crossencoder_output_eval.json",
        "sapbert" : f"/home2/cye73/results2/sapbert/sapbert_{dataset_name}_real.json",
        # "scispacy": f"/home2/cye73/results2/scispacy/{dataset_name}.json",
    }}

evaluator = Evaluate(dataset_names, model_names, path_to_result,
                     path_to_abbrev
                     )
evaluator.load_results()
evaluator.process_datasets()
evaluator.evaluate(
    eval_strategies=['basic']
                   )
# evaluator.plot_results(
#     # eval_strategies=['basic']
#     )

Skipping model arboel_biencoder for dataset medmentions_st21pv because the file is missing or the path is invalid.
dataset : medmentions_st21pv, model : arboel_crossencoder
dataset : medmentions_st21pv, model : sapbert


  0%|          | 0/1 [00:00<?, ?it/s]/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py:251: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda x: min_hit_index(x[0], x[1], eval_mode=eval_mode), axis=1


Eval Strategy: basic
medmentions_st21pv


/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py:251: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda x: min_hit_index(x[0], x[1], eval_mode=eval_mode), axis=1
/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py:251: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda x: min_hit_index(x[0], x[1], eval_mode=eval_mode), axis=1
/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py:251: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value

In [20]:
# device_1 = torch.device("cuda:0")
# device_2 = torch.device("cuda:1")

# llm = LLM(
# model=model1,
# tensor_parallel_size=1,
# device=device_1)

# llm2 = LLM(
# model=model2,
# tensor_parallel_size=1,
# device=device_2)

In [58]:
# EL_model = "crossencoder"
# EL_model = "sapbert"
number_candidates = 20
results = evaluator.full_results["basic"][f"{dataset_name}"]
cols = ['document_id', 'offsets', 'deabbreviated_text', 'db_ids', 'mention_id', 'joined_offsets', 
        # 'arboel_biencoder_resolve_abbrev', 'arboel_biencoder_resolve_abbrev_min_hit_index', 
        # 'arboel_crossencoder_resolve_abbrev', 'arboel_crossencoder_resolve_abbrev_min_hit_index', 
        'sapbert_resolve_abbrev', 'sapbert_resolve_abbrev_min_hit_index', 
        # 'scispacy_resolve_abbrev', 'scispacy_resolve_abbrev_min_hit_index'
        ]
filtered_results = results[cols].rename(columns={
        # 'arboel_biencoder_resolve_abbrev': 'biencoder_candidates',
        # 'arboel_crossencoder_resolve_abbrev': 'crossencoder_candidates',
        'sapbert_resolve_abbrev': 'sapbert_candidates',
        # 'arboel_biencoder_resolve_abbrev_min_hit_index': 'biencoder_hit_index',
        # 'arboel_crossencoder_resolve_abbrev_min_hit_index': 'crossencoder_hit_index',
        'sapbert_resolve_abbrev_min_hit_index': 'sapbert_hit_index',
#  'scispacy_resolve_abbrev_min_hit_index': 'scispacy_hit_index',
})
filtered_results = filtered_results[filtered_results[f'{EL_model}_hit_index'] < number_candidates]

# filtered_results = filtered_results[filtered_results['crossencoder_candidates'].apply(lambda x: len(x[0]) != 0 if x else False)]

# filtered_results = filtered_results.iloc[:500]
filtered_results

,document_id,offsets,deabbreviated_text,db_ids,mention_id,joined_offsets,sapbert_candidates,sapbert_hit_index
0,25847295,"[[34, 43]]",apoptosis,[UMLS:C0162638],25847295.1,"34,43","[[UMLS:C0162638], [UMLS:C1516044], [UMLS:C1159...",0
1,25847295,"[[55, 65]]",PC12 cells,[UMLS:C0085262],25847295.2,"55,65","[[UMLS:C0085262], [UMLS:C4054122], [UMLS:C0376...",0
2,25847295,"[[137, 144]]",present,[UMLS:C0150312],25847295.3,"137,144","[[UMLS:C0150312], [UMLS:C0449450], [UMLS:C0392...",0
3,25847295,"[[206, 219]]",toxic effects,[UMLS:C0600688],25847295.4,"206,219","[[UMLS:C0600688], [UMLS:C0496113], [UMLS:C1256...",0
4,25847295,"[[259, 268]]",Apoptosis,[UMLS:C0162638],25847295.5,"259,268","[[UMLS:C0162638], [UMLS:C1516044], [UMLS:C1159...",0
...,...,...,...,...,...,...,...,...
40135,28550165,"[[1603, 1609]]",CLASP2,[UMLS:C1566863],28550165.87,"1603,1609","[[UMLS:C1424727], [UMLS:C1424735], [UMLS:C1412...",4
40136,28550165,"[[1624, 1631]]",tubulin,[UMLS:C0041348],28550165.88,"1624,1631","[[UMLS:C0041348], [UMLS:C2937342], [UMLS:C0085...",0
40138,28550165,"[[1665, 1695]]",microtubule-associated protein,[UMLS:C0026045],28550165.90,"1665,1695","[[UMLS:C0026045], [UMLS:C0024772], [UMLS:C1180...",0
40140,28550165,"[[1813, 1825]]",microtubules,[UMLS:C0026046],28550165.92,"1813,1825","[[UMLS:C0026046], [UMLS:C2336504], [UMLS:C0544...",0


In [22]:
# new_res = filtered_results[(filtered_results['sapbert_hit_index']!=0) & 
#                            (filtered_results['sapbert_hit_index']>filtered_results['crossencoder_hit_index'])&
#                            (filtered_results['sapbert_hit_index'] < 64)]
# new_res

In [23]:
# cands = ["MESH:D003919", "MESH:D003923", "MESH:D011254","MESH:D048909","MESH:C564219","MESH:D006029","MESH:C562774","MESH:C565632","MESH:C563322"]
# cands2 = ["MESH:D020790", "MESH:D018500", "MESH:D003919"]

# for cand in cands :
#     print(f"{cand} :", ontology2.entities[cand].name)
# print("--------")
# for cand in cands2 :
#     print(f"{cand} :", ontology2.entities[cand].name)


In [24]:
total_mentions = len(results)
total_mentions

11624

# Part with GPT

### Train set

In [25]:

train_mentions = []
train_mention2context = {}
train_mention2gold = {}
train_mention2text = {}
for idx, row in train_df.iterrows():
    train_mention2gold[row['mention_id']] = row['db_ids']
    train_mentions.append(row['mention_id'])
    train_mention2text[row['mention_id']] = row['deabbreviated_text']
    train_mention2context[row["mention_id"]] = row["limited_contextualized_mention"]

# train_mention2gold

### Test set

In [26]:
mention2context = {}
for idx, row in test_df.iterrows():
    mention2context[row["mention_id"]] = row["limited_contextualized_mention"]

mentions = []
mention2biencoder_candidates = {}
mention2crossencoder_candidates = {}
mention2sapbert_candidates = defaultdict(list)
mention2gold = {}
mention2hit = {}
mention2text = {}
cui2name = {}
for idx, row in filtered_results.iterrows():
    # Only consider row if hit_index < max number of candidates
    if row[f'{EL_model}_hit_index'] < number_candidates:
        if EL_model == "biencoder":
            mention2biencoder_candidates[row['mention_id']] = [el[0] for el in row['biencoder_candidates'][:number_candidates]]
        
        elif EL_model == "crossencoder":
            mention2crossencoder_candidates[row['mention_id']] = [el[0] for el in row['crossencoder_candidates'][:number_candidates]]
        
        elif EL_model == "sapbert":
            mention2sapbert_candidates[row['mention_id']] = [el[0] for el in row['sapbert_candidates'][:number_candidates]] 

        elif EL_model == "scispacy":
            mention2sapbert_candidates[row['mention_id']] = [el[0] for el in row['scispacy_candidates'][:number_candidates]] 
        # All this trouble so that we keep the order of the candidates
        # This strategy increases the number of candidates past the number_candidates limit so we will need to filter them out later
        
        mention2gold[row['mention_id']] = row['db_ids']
        mentions.append(row['mention_id'])
        mention2text[row['mention_id']] = row['deabbreviated_text']
    
    
        mention2hit[row['mention_id']] = row['biencoder_hit_index']

In [27]:
number_hits_biencoder = 0
number_hits_crossencoder = 0
number_hits_sapbert = 0
if EL_model == "biencoder":
    number_hits_biencoder = number_hit(
        mention2biencoder_candidates, mention2gold, number_candidates
    )
    print("number hits biencoder :", number_hits_biencoder)
    biencoder_results = compute_recall(number_hits_biencoder, number_candidates, total_mentions)
    print("Biencoder:")
    for i, (unnormalized, normalized) in enumerate(biencoder_results[:10]):
        print(
            f"recall {i+1}: Normalized = {normalized:.4f}, Unnormalized = {unnormalized:.4f}"
        )

elif EL_model == "crossencoder":
    number_hits_crossencoder = number_hit(
        mention2crossencoder_candidates, mention2gold, number_candidates
    )
    print("number hits crossencoder :", number_hits_crossencoder)
    crossencoder_results = compute_recall(number_hits_crossencoder, number_candidates, total_mentions)
    print("Crossencoder:")
    for i, (unnormalized, normalized) in enumerate(crossencoder_results[:10]):
        print(
            f"recall {i+1}: Normalized = {normalized:.4f}, Unnormalized = {unnormalized:.4f}"
        )

elif EL_model == "sapbert":
    number_hits_sapbert = number_hit(
        mention2sapbert_candidates, mention2gold, number_candidates
    )
    print("number hits sapbert :", number_hits_sapbert)

    sapbert_results = compute_recall(number_hits_sapbert, number_candidates, total_mentions)
    print("Sapbert:")
    for i, (unnormalized, normalized) in enumerate(sapbert_results[:10]):
        print(
            f"recall {i+1}: Normalized = {normalized:.4f}, Unnormalized = {unnormalized:.4f}"
        )

number hits crossencoder : {1: 9051, 2: 640, 3: 138, 4: 64, 5: 21, 6: 19, 7: 11, 8: 13, 9: 5, 10: 5, 11: 5, 12: 7, 13: 2, 14: 3, 15: 1, 16: 3, 17: 1, 18: 0, 19: 2, 20: 0}
Crossencoder:
recall 1: Normalized = 0.9059, Unnormalized = 0.7786
recall 2: Normalized = 0.9700, Unnormalized = 0.8337
recall 3: Normalized = 0.9838, Unnormalized = 0.8456
recall 4: Normalized = 0.9902, Unnormalized = 0.8511
recall 5: Normalized = 0.9923, Unnormalized = 0.8529
recall 6: Normalized = 0.9942, Unnormalized = 0.8545
recall 7: Normalized = 0.9953, Unnormalized = 0.8555
recall 8: Normalized = 0.9966, Unnormalized = 0.8566
recall 9: Normalized = 0.9971, Unnormalized = 0.8570
recall 10: Normalized = 0.9976, Unnormalized = 0.8575


In [28]:
system_instructions = """You are a professional data annotator and curator.
Your task is to identify the correct entity for a given mention based on the provided context and the descriptions of {number_candidates} candidate entities."""

system_instructions_recall = """You are a professional data annotator and curator.
Your task is to rank the candidate entities from best to worst for a given mention based on the provided context and the descriptions of each candidate entities."""


### Single prompt for error analysis

In [29]:
# entities_cui = list(ontology2.entities)
# entities_cui2data = get_candidates_data_v2(entities_cui, ontology2)
# entities_data2cui = {v: k for k, v in entities_cui2data.items()}
# entities_data2cui
# entities_data = list(entities_cui2data.values())
# entities_data

In [35]:
# from sentence_transformers import SentenceTransformer

# # model = SentenceTransformer('princeton-nlp/sup-simcse-bert-base-uncased')
# # model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# from transformers import AutoModel
# model = AutoModel.from_pretrained('nvidia/NV-Embed-v1', trust_remote_code=False)
# model.to(device_1)

# # Generate embeddings for the corpus
# entities_embeddings = model.encode(entities_data, convert_to_tensor=True)

# entities_embeddings = entities_embeddings.cpu().detach().numpy()

# embedding_dimension = entities_embeddings.shape[1]

# # Create the HNSW index with the correct arguments
# M = 32  # Number of neighbors in the HNSW graph
# index_ent = faiss.IndexHNSWFlat(embedding_dimension, M)

# # Normalize the corpus embeddings if using cosine similarity
# faiss.normalize_L2(entities_embeddings)

# # Add the embeddings to the index
# index_ent.add(entities_embeddings)

# # Print the number of sentences added to the index
# print(f"Number of sentences in the index: {index_ent.ntotal}")

In [36]:
# mention = "9973276.8"

# text = mention2text[mention]
# context = mention2context[mention]

# topk = topk_entities(model = model,
#               index_entity=index_ent,
#               query=context,
#               entity_corpus = entities_data,
#               entities_data2cui = entities_data2cui,
#               k=10,
#               )

# data = get_candidates_data(mention2gold[mention], ontology2)


# print("Mention ID :", mention)
# print("Text :", text)
# for i, entity_cui in enumerate(topk):
#     print(f"Candidate {i+1} :", entity_cui, entities_cui2data[entity_cui])
# print("Gold CUI :", mention2gold[mention], "Name:", data)

In [37]:
# total_hits = 0
# for i, mention_id in enumerate(mentions):
#     gold_cui = mention2gold[mention_id][0]
#     candidates = mention2crossencoder_candidates[mention_id][:5]
#     if gold_cui in candidates:
#         total_hits += 1
# total_hits

In [38]:
# total_hits = 0
# for i, mention_id in enumerate(mentions):
#     text = mention2text[mention_id]
#     context = mention2context[mention_id]

#     topk = topk_entities(model = model,
#                 index_entity=index_ent,
#                 query=context,
#                 entity_corpus = entities_data,
#                 entities_data2cui = entities_data2cui,
#                 k=5,
#                 )

#     data = get_candidates_data(mention2gold[mention_id], ontology2)
#     gold_cui = mention2gold[mention_id][0]
    
#     candidates = list(set(mention2crossencoder_candidates[mention_id][:5] + topk))
#     if gold_cui in candidates:
#         total_hits += 1
        
#     # if i < 2 :
#     #     print("Text :", text)
#     #     for i, entity_cui in enumerate(topk):
#     #         print(f"Candidate {i+1} :", entity_cui, entities_cui2data[entity_cui])
#     #     print("Gold CUI :", mention2gold[mention], "Name:", data)

In [39]:
# total_hits

In [40]:
# hit = defaultdict(int)
# for i, mention in enumerate(mentions):
#     text = mention2text[mention]
#     context = mention2context[mention]

#     topk = topk_entities(model = model,
#                 index_entity=index_ent,
#                 query=text,
#                 entity_corpus = entities_data,
#                 entities_data2cui = entities_data2cui,
#                 k=1000,
#                 )

#     data = get_candidates_data(mention2gold[mention], ontology2)
#     gold_cui = list(data.keys())[0]
    
#     if gold_cui in topk:
#         ind = topk.index(gold_cui)
#         hit[ind] += 1
        
#     # if i < 2 :
#     #     print("Text :", text)
#     #     for i, entity_cui in enumerate(topk):
#     #         print(f"Candidate {i+1} :", entity_cui, entities_cui2data[entity_cui])
#     #     print("Gold CUI :", mention2gold[mention], "Name:", data)

In [41]:
# print("Numbers of hit:", sum(hit.values()))
# print(hit)

In [42]:
# sorted_hits = sorted(hit.keys())
# sorted_hit_dict = {key: hit[key] for key in sorted_hits}
# print(sorted_hit_dict)

In [43]:
# # mention = '9931324.9' # "9973276.8"
# mention_id = "9288106.17"
# # i = 39
# # mention = mentions[i]
# mention_name = mention2text[mention_id]
# context = mention2context[mention_id]
# candidates = get_candidates_data(candidates = mention2biencoder_candidates[mention_id], 
#                                  ontology = ontology2)
# topk = topk_examples(model = model,
#                     index = index,
#                     query=context, 
#                     corpus=corpus, 
#                     TrainMap_context2mention=TrainMap_context2mention, 
#                     train_mention2text=train_mention2text, 
#                     train_mention2gold=train_mention2gold, 
#                     ontology=ontology2, 
#                     k=3)
# # random.shuffle(candidates)
# # candidates = get_candidates_data_v2(mention2candidates[mentions[i]])

# text = generate_prompt_text(
#             mention=mention_name,
#             context=context,
#             candidates=candidates,
#             topk_examples=topk,
#             reasoning=False,
#             recall=False,
#             recall_k=5,
#             analysis_version="v1",
#             analysis=None,
#             mention_id=mention_id,
#         )

# pred_cui = prompt_gpt(prompt=text,
#                       system_instructions=system_instructions, 
#                       gpt_version = "gpt-4o-mini")
# # model = "gpt-3.5-turbo-0125"
# # model = "gpt-4o-2024-08-06"
# print("Mention ID :", mention)
# print("Context :", context)
# print("Top k examples :", topk)
# print("Text :", text)
# print("Gold CUI :", mention2gold[mention])

# print("Predicted CUI :", pred_cui)

In [44]:
# results = evaluate_vllm(llm = "gpt-4o-mini",
#                     nlp_model=model,
#                     tokenizer=None,
#                     index=index,
#                     system_instructions=system_instructions,
#                     ontology=ontology2, 
#                     mentions=mentions[:5], 
#                     corpus=corpus,
#                     mention2text=mention2text,
#                     mention2context=mention2context,
#                     mention2biencoder_candidates=mention2biencoder_candidates,   
#                     TrainMap_context2mention=TrainMap_context2mention,
#                     train_mention2text=train_mention2text,
#                     train_mention2gold=train_mention2gold,
#                     k=3, 
#                     sampling_params=None,
#                     reasoning=False,
#                     analysis=None,
#                     analysis_version="v1",
#                     recall=False,
#                     recall_k=5,)
# results

In [ ]:
score = scoring(results=results, mention2gold=mention2gold)
score


In [31]:
### SAVE THE RESULTS ! IT'S EXPENSIVE TO RUN THE MODEL !!!

# with open("gpt4o_results.json", "w") as f:
#     json.dump(results, f, indent=4)

with open("gpt4o_results.json", "r") as f:
    results = json.load(f)

# results

In [ ]:
error_mentions = error_analysis(results = results, 
                                ontology = ontology2, 
                                mention2gold = mention2gold, 
                                mention2context = mention2context)
error_mentions

# Evaluation with Mixtral

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

mixtral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
mixtral_model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")


### ICL

In [44]:
def prompt_mistral(mention, context, candidates, topk_examples):
    ''' 
    Equivalent of the prompt function for the Mistral model.
    '''
    
    prompt_text = f"""
    You are a professional data annotator and curator.
    Your task is to identify the correct entity for a given mention based on the provided context and the descriptions of {number_candidates} candidate entities. \n
    
    Here are a few examples : \n
    {topk_examples} \n

    This is the specific mention that needs to be linked to the correct entity : {mention} \n
    
    This is the context where the mention appears : \n
    {context} \n
    
    These are the candidate entities to choose from: \n
    {candidates} \n
    
    You must provide an answer among the candidates. \n
    
    Return the answer in the following format : CUI
    Do not add any explanations!
    """

    # Tokenize input
    inputs = mixtral_tokenizer(prompt_text, return_tensors="pt", max_length=2048, truncation=True)

    # Generate output (you can adjust `max_new_tokens` to limit the output)
    with torch.no_grad():
        outputs = mixtral_model.generate(**inputs, max_new_tokens=300, temperature=0.0)

    # Decode the generated tokens
    generated_text = mixtral_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the CUI answer (since the model generates the entire text, you might need to clean it)
    return generated_text.strip()

In [ ]:
# mention = '9931324.9' # "9973276.8"
mention = "9888388.6"
# i = 39
# mention = mentions[i]
text = mention2text[mention]
context = mention2context[mention]
candidates = get_candidates_data(candidates=mention2biencoder_candidates[mention], 
                                 ontology=ontology2)
topk = topk_examples(query=context, 
                        corpus=corpus, 
                        TrainMap_context2mention=TrainMap_context2mention, 
                        train_mention2text=train_mention2text, 
                        train_mention2gold=train_mention2gold, 
                        ontology=ontology2, 
                        k=3)

pred_cui = prompt_mistral(text, context, candidates, topk_examples=topk)

print("Predicted CUI :", pred_cui)

# Flan T5 model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load Flan-T5 model and tokenizer
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-xxl")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-xxl")


In [29]:
def prompt_flan_t5(mention, context, candidates, topk_examples):
    prompt_text = f"""
    You are a professional data annotator and curator.
    Your task is to identify the correct entity for a given mention based on the provided context and the descriptions of 64 candidate entities.

    Here are a few examples: \n
    {topk_examples} \n

    This is the specific mention that needs to be linked to the correct entity: {mention} \n

    This is the context where the mention appears : \n
    {context} \n
    
    These are the candidate entities to choose from: \n
    {candidates} \n
    
    You must provide an answer among the candidates. \n

    Return the answer in the following format: CUI
    Do not add any explanations!
    """

    inputs = flan_tokenizer(prompt_text, return_tensors="pt", max_length=2048, truncation=True)

    outputs = flan_model.generate(**inputs, max_new_tokens=100, temperature=0.0)
    generated_text = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text.strip()


In [ ]:
mention = "9988281.10"
text = mention2text[mention]
context = mention2context[mention]
candidates = get_candidates_data(mention2biencoder_candidates[mention], ontology2)
topk = topk_examples(query=context, 
                        corpus=corpus, 
                        TrainMap_context2mention=TrainMap_context2mention, 
                        train_mention2text=train_mention2text, 
                        train_mention2gold=train_mention2gold, 
                        ontology=ontology2, 
                        k=3)

pred_cui = prompt_flan_t5(mention = text, context = context, candidates=candidates, topk_examples=topk)

print("Mention ID :", mention)
print("Context :", context)
print("Top k examples :", topk)
print("Text :", mention2text[mention])
print("Gold CUI :", mention2gold[mention])
print("Predicted CUI :", pred_cui)

In [ ]:
CUIs = [v[0] for k, v in mention2gold.items()] # List of all gold CUIs from the test set (reduced one = only with hit_index < nb_candidates)

cui2name = get_candidates_name(CUIs, ontology2)
name2cui = {v: k for k, v in cui2name.items()}

name2cui[pred_cui]

# vllm
# LLAMA
Llama-3.1-8B-UltraMedical (71%)

Bio-Medical-Llama-3-8B (41%)

Laim/Llama-3.1-MedPalm2-imitate-8B-Instruct (17%)

meta-llama/Meta-Llama-3.1-8B-Instruct (79%) (91% for k=10)

mistralai/Mistral-7B-Instruct-v0.3 (82%) (84% for k=10)

mistralai/Mistral-Nemo-Instruct-2407 (90%)

ISTA-DASLab/Mixtral-8x7B-Instruct-v0_1-AQLM-2Bit-1x16-hf

In [ ]:
# llm = LLM(
#     model="meta-llama/Meta-Llama-3.1-8B-Instruct",
#     enforce_eager=True,
# )

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
device_1 = torch.device("cuda:0")  # First GPU (GPU 0)
device_2 = torch.device("cuda:1")  # Second GPU (GPU 1)

llm = LLM(
model="Qwen/Qwen2.5-7B-Instruct",
tensor_parallel_size=1,
dtype = "half",
gpu_memory_utilization=0.85,
max_logprobs=1000,
device=device_1,
max_model_len = 30000)

tokenizer = llm.get_tokenizer()

In [27]:
# llm2 = LLM(
# model="mistralai/Mistral-7B-Instruct-v0.3",
# tensor_parallel_size=1,
# dtype = "half",
# gpu_memory_utilization=0.9,
# max_logprobs=1000,
# device=device_2)

In [28]:
# torch.cuda.set_device(0)
# torch.cuda.empty_cache()

In [ ]:
mention_id = "9448273.1"
mention_name = mention2text[mention_id]
context = mention2context[mention_id]
candidates = get_candidates_data(mention2biencoder_candidates[mention_id], ontology2)
topk = topk_examples(model = model, # sentence transformer model
                    index=index,
                    query=context,
                    corpus=corpus,
                    TrainMap_context2mention=TrainMap_context2mention,
                    train_mention2text=train_mention2text,
                    train_mention2gold=train_mention2gold,
                    ontology=ontology2,
                    k=3)

prompt = generate_prompt_text(mention = mention_name,
                              context=context,
                              candidates=candidates,
                              topk_examples=topk,
                              recall=False,
                              recall_k=5,
                              mention_id=mention_id)

# Generate the prediction using LLaMA 8B
pred_cui = prompt_vllm(prompt = prompt,
                       system_instructions=system_instructions,
                       llm=llm,
                       tokenizer=tokenizer,
                       sampling_params=sampling_params)
answer = extract_last_cui(pred_cui)

# Output the results
print("Mention ID:", mention_id)
print("Context:", context)
print("Top k examples:", topk)
print("Text:", mention2text[mention_id])
print("Gold CUI:", mention2gold[mention_id])
print("Predicted CUI:", pred_cui)
print("Final answer :", answer)

In [ ]:
print(prompt)

In [ ]:
results = evaluate_vllm(llm=llm,
                        nlp_model=model,
                        tokenizer=tokenizer,
                        index=index,
                        system_instructions=system_instructions,
                        mentions=mentions,
                        ontology=ontology2,
                        corpus=corpus,
                        mention2context=mention2context,
                        mention2biencoder_candidates=mention2biencoder_candidates,
                        mention2text=mention2text,
                        TrainMap_context2mention=TrainMap_context2mention,
                        train_mention2text=train_mention2text,
                        train_mention2gold=train_mention2gold,
                        k=3,
                        sampling_params=sampling_params)
results

In [ ]:
score = scoring(results=results, 
                mention2gold=mention2gold)
score

## Result : With reasoning

In [28]:
### SAVE THE RESULTS (It's long to run the model 26mins)
    
# with open("Meta-Llama-3.1-8B-Instruct_k=10_reasoning_results.json", "w") as f:
#     json.dump(results, f, indent=4)

with open("data/biencoder/reasoning2/Meta-Llama-3.1-8B-Instruct_k=3_reasoning_results.json", "r") as f:
    results1r = json.load(f)
    
with open("data/biencoder/reasoning2/Meta-Llama-3.1-8B-Instruct_k=5_reasoning_results.json", "r") as f:
    results2r = json.load(f)
    
with open("data/biencoder/reasoning2/Meta-Llama-3.1-8B-Instruct_k=10_reasoning_results.json", "r") as f:
    results3r = json.load(f)
    
    
###############################################################  
with open("data/biencoder/reasoning2/Mistral-7B-Instruct-v0.3_k=3_reasoning_results.json", "r") as f:
    results4r = json.load(f)

with open("data/biencoder/reasoning2/Mistral-7B-Instruct-v0.3_k=5_reasoning_results.json", "r") as f:
    results5r = json.load(f)
    
with open("data/biencoder/reasoning2/Mistral-7B-Instruct-v0.3_k=10_reasoning_results.json", "r") as f:
    results6r = json.load(f)


###############################################################
with open("data/biencoder/reasoning2/Mistral-Nemo-Instruct-2407_k=3_reasoning_results.json", "r") as f:
    results7r = json.load(f)

with open("data/biencoder/reasoning2/Mistral-Nemo-Instruct-2407_k=5_reasoning_results.json", "r") as f:
    results8r = json.load(f)

with open("data/biencoder/reasoning2/Mistral-Nemo-Instruct-2407_k=10_reasoning_results.json", "r") as f:
    results9r = json.load(f)
    
    
###############################################################
with open("data/biencoder/reasoning2/Qwen2.5-7B-Instruct_k=3_reasoning_results.json", "r") as f:
    results10r = json.load(f)

# with open("data/biencoder/reasoning2/Qwen2.5-7B-Instruct_k=5_reasoning_results.json", "r") as f:
#     results11r = json.load(f)

# with open("data/biencoder/reasoning2/Qwen2.5-7B-Instruct_k=10_reasoning_results.json", "r") as f:
#     results12r = json.load(f)
    
    
###############################################################
with open("data/biencoder/reasoning2/Qwen2.5-14B-Instruct_k=3_reasoning_results.json", "r") as f:
    results13r = json.load(f)

# with open("data/biencoder/reasoning2/Qwen2.5-14B-Instruct_k=5_reasoning_results.json", "r") as f:
#     results14r = json.load(f)

# with open("data/biencoder/reasoning2/Qwen2.5-14B-Instruct_k=10_reasoning_results.json", "r") as f:
#     results15r = json.load(f)

In [ ]:
# for mention_id in results.copy() :
#     if results[mention_id] is None:
#         del results[mention_id]

score1r = scoring(results = results1r, mention2gold = mention2gold)
print("Meta-Llama-3.1-8B-Instruct_k=3_reasoning_results :", score1r)
score2r = scoring(results = results2r, mention2gold = mention2gold)
print("Meta-Llama-3.1-8B-Instruct_k=5_reasoning_results:", score2r)
score3r = scoring(results = results3r, mention2gold = mention2gold)
print("Meta-Llama-3.1-8B-Instruct_k=10_reasoning_results:", score3r)

print("---------------------------------")

score4r = scoring(results = results4r, mention2gold = mention2gold)
print("Mistral-7B-Instruct-v0.3_k=3_reasoning_results :", score4r)
score5r = scoring(results = results5r, mention2gold = mention2gold)
print("Mistral-7B-Instruct-v0.3_k=5_reasoning_results :", score5r)
score6r = scoring(results = results6r, mention2gold = mention2gold)
print("Mistral-7B-Instruct-v0.3_k=10_reasoning_results :", score6r)

print("---------------------------------")

score7r = scoring(results = results7r, mention2gold = mention2gold)
print("Mistral-Nemo-Instruct-2407_k=3_reasoning_results :", score7r)
score8r = scoring(results = results8r, mention2gold = mention2gold)
print("Mistral-Nemo-Instruct-2407_k=5_reasoning_results:", score8r)
score9r = scoring(results = results9r, mention2gold = mention2gold)
print("Mistral-Nemo-Instruct-2407_k=10_reasoning_results:", score9r)

print("---------------------------------")

score10r = scoring(results = results10r, mention2gold = mention2gold)
print("Qwen2.5-7B-Instruct_k=3_results :", score10r)
# score11r = scoring(results = results11r, mention2gold = mention2gold)
# print("Qwen2.5-7B-Instruct_k=5_results:", score11r)
# score12r = scoring(results = results12r, mention2gold = mention2gold)
# print("Qwen2.5-7B-Instruct_k=10_results:", score12r)

print("---------------------------------")

score13r = scoring(results = results13r, mention2gold = mention2gold)
print("Qwen2.5-14B-Instruct_k=3_results :", score13r)
# score14r = scoring(results = results14r, mention2gold = mention2gold)
# print("Qwen2.5-14B-Instruct_k=5_results:", score14r)
# score15r = scoring(results = results15r, mention2gold = mention2gold)
# print("Qwen2.5-14B-Instruct_k=10_results:", score15r)

In [ ]:
error_mentions = error_analysis(results = results13r, 
                                ontology = ontology2, 
                                mention2gold = mention2gold, 
                                mention2context = mention2context)
error_mentions

In [ ]:
mention_id = '9988281.10'
print(results13r[mention_id]["explanation"])

## Result : Without reasoning

In [32]:
# with open("data/biencoder/default2/Meta-Llama-3.1-8B-Instruct_k=3_results.json", "r") as f:
#     results1 = json.load(f)
    
# with open("data/biencoder/default2/Meta-Llama-3.1-8B-Instruct_k=5_results.json", "r") as f:
#     results2 = json.load(f)
    
# with open("data/biencoder/default2/Meta-Llama-3.1-8B-Instruct_k=10_results.json", "r") as f:
#     results3 = json.load(f)
    
    
# ###############################################################  
# with open("data/biencoder/default2/Mistral-7B-Instruct-v0.3_k=3_results.json", "r") as f:
#     results4 = json.load(f)

# with open("data/biencoder/default2/Mistral-7B-Instruct-v0.3_k=5_results.json", "r") as f:
#     results5 = json.load(f)
    
# with open("data/biencoder/default2/Mistral-7B-Instruct-v0.3_k=10_results.json", "r") as f:
#     results6 = json.load(f)


# ###############################################################
# with open("data/biencoder/default2/Mistral-Nemo-Instruct-2407_k=3_results.json", "r") as f:
#     results7 = json.load(f)

# with open("data/biencoder/default2/Mistral-Nemo-Instruct-2407_k=5_results.json", "r") as f:
#     results8 = json.load(f)

# with open("data/biencoder/default2/Mistral-Nemo-Instruct-2407_k=10_results.json", "r") as f:
#     results9 = json.load(f)
    
# ###############################################################
# with open("data/biencoder/default2/Qwen2.5-7B-Instruct_k=3_results.json", "r") as f:
#     results10 = json.load(f)

# with open("data/biencoder/default2/Qwen2.5-7B-Instruct_k=5_results.json", "r") as f:
#     results11 = json.load(f)

# with open("data/biencoder/default2/Qwen2.5-7B-Instruct_k=10_results.json", "r") as f:
#     results12 = json.load(f)
    
    
# ###############################################################
# with open("data/biencoder/default2/Qwen2.5-14B-Instruct_k=3_results.json", "r") as f:
#     results13 = json.load(f)

# with open("data/biencoder/default2/Qwen2.5-14B-Instruct_k=5_results.json", "r") as f:
#     results14 = json.load(f)

# with open("data/biencoder/default2/Qwen2.5-14B-Instruct_k=10_results.json", "r") as f:
#     results15 = json.load(f)

In [ ]:
# score1 = scoring(results = results1, mention2gold = mention2gold)
# print("data/default/Meta-Llama-3.1-8B-Instruct_k=3_results :", score1)
# score2 = scoring(results = results2, mention2gold = mention2gold)
# print("data/default/Meta-Llama-3.1-8B-Instruct_k=5_results:", score2)
# score3 = scoring(results = results3, mention2gold = mention2gold)
# print("data/default/Meta-Llama-3.1-8B-Instruct_k=10_results:", score3)

# print("---------------------------------")

# score4 = scoring(results = results4, mention2gold = mention2gold)
# print("data/default/Mistral-7B-Instruct-v0.3_k=3_results :", score4)
# score5 = scoring(results = results5, mention2gold = mention2gold)
# print("data/default/Mistral-7B-Instruct-v0.3_k=5_results :", score5)
# score6 = scoring(results = results6, mention2gold = mention2gold)
# print("data/default/Mistral-7B-Instruct-v0.3_k=10_results :", score6)

# print("---------------------------------")

# score7 = scoring(results = results7, mention2gold = mention2gold)
# print("data/default/Mistral-Nemo-Instruct-2407_k=3_results :", score7)
# score8 = scoring(results = results8, mention2gold = mention2gold)
# print("data/default/Mistral-Nemo-Instruct-2407_k=5_results:", score8)
# score9 = scoring(results = results9, mention2gold = mention2gold)
# print("data/default/Mistral-Nemo-Instruct-2407_k=10_results:", score9)

# print("---------------------------------")

# score10 = scoring(results = results10, mention2gold = mention2gold)
# print("data/default/Qwen2.5-7B-Instruct_k=3_results :", score10)
# score11 = scoring(results = results11, mention2gold = mention2gold)
# print("data/default/Qwen2.5-7B-Instruct_k=5_results:", score11)
# score12 = scoring(results = results12, mention2gold = mention2gold)
# print("data/default/Qwen2.5-7B-Instruct_k=10_results:", score12)

# print("---------------------------------")

# score13 = scoring(results = results13, mention2gold = mention2gold)
# print("data/default/Qwen2.5-14B-Instruct_k=3_results :", score13)
# score14 = scoring(results = results14, mention2gold = mention2gold)
# print("data/default/Qwen2.5-14B-Instruct_k=5_results:", score14)
# score15 = scoring(results = results15, mention2gold = mention2gold)
# print("data/default/Qwen2.5-14B-Instruct_k=10_results:", score15)



In [34]:
# def pooled_scoring(results, mention2gold):
#     """
#     Return the score of the model.
#     -------
#     results : list of dictionaries [{mention_id : {"predicted" : predicted CUI, "explanation" : explanation}}, ...]
#     mention2gold : dictionary {mention_id : gold CUI or list of gold CUIs}
#     """
#     score = 0
#     found_mentions = set()  # To track which mentions have already been correctly predicted

#     # Iterate over each set of predictions (each result_i)
#     for result in results:
#         for mention_id, value in result.items():
#             if mention_id in found_mentions:
#                 continue  # Skip if this mention was already counted
            
#             predicted_cui = value["predicted"]
#             gold_cuis = mention2gold[mention_id]  # Gold CUI or list of CUIs
            
#             # Ensure gold_cuis is a list for consistent comparison
#             if not isinstance(gold_cuis, list):
#                 gold_cuis = [gold_cuis]

#             # Check if the predicted CUI is correct
#             if predicted_cui in gold_cuis:
#                 score += 1  # Increment score only once per mention_id
#                 found_mentions.add(mention_id)  # Mark this mention as found

#     return score / len(mention2gold)

In [ ]:
# results = [
#     results1, 
#     # results2, 
#     # results3,    
#     results4, 
#     # results5, 
#     # results6,
#     results7, 
#     # results8, 
#     # results9,
#     results10, 
#     # results11, 
#     # results12,
#     results13, 
#     # results14, 
#     # results15,
#     ]
# score = pooled_scoring(results, mention2gold)
# score

In [ ]:
# results_r = [
#     results1r, 
#     # results2r, 
#     # results3r,    
#     results4r, 
#     # results5r, 
#     # results6r,
#     results7r, 
#     # results8r, 
#     # results9r,
#     results10r, 
#     # results11r, 
#     # results12r,
#     results13r, 
#     # results14r, 
#     # results15r,
#     ]
# score_r = pooled_scoring(results_r, mention2gold)
# score_r

In [42]:
# len(results)
# len(mention2gold)

In [ ]:
# error_mentions = error_analysis(results = results2, 
#                                 ontology = ontology2, 
#                                 mention2gold = mention2gold, 
#                                 mention2context = mention2context)
# error_mentions

In [36]:
# def parse_log_file_acc(log_file_path, start_line, end_line):
#     results = {}

#     with open(log_file_path, 'r') as file:
#         lines = file.readlines()

#     # We will only process the lines between start_line and end_line
#     for i in range(start_line - 1, end_line):  # Convert to 0-based index
#         line = lines[i].strip()
#         if "mention ID" in line:
#             # Extract mention ID and LLM answer
#             mention_id = line.split(' || ')[0].split(' : ')[1].strip()
#             llm_answer = line.split(' || ')[1].split(' : ')[1].strip()
#             # Store in the dictionary
#             results[mention_id] = {"predicted": llm_answer, "explanation": llm_answer}

#     return results


# import re
# import json

# def parse_log_file(file_path):
#     results = {}
#     current_mention_id = None
    
#     with open(file_path, 'r') as file:
#         for line in file:
#             mention_id_match = re.search(r'mention ID : (\d+\.\d+)', line)
#             if mention_id_match:
#                 current_mention_id = mention_id_match.group(1)
#                 llm_answer_str = line.split("LLM answer :")[1].strip()
                
#                 # Handle JSON format with backticks
#                 json_match = re.search(r'```json\s*(.*?)\s*(?:```|\Z)', llm_answer_str, re.DOTALL)
#                 if json_match:
#                     try:
#                         llm_answer = json.loads(json_match.group(1))
#                     except json.JSONDecodeError:
#                         # If JSON parsing fails, fall back to regex
#                         llm_answer = re.findall(r'"([^"]*)"', json_match.group(1))
#                 else:
#                     # If no JSON format, use regex to extract list items
#                     llm_answer = re.findall(r'"([^"]*)"', llm_answer_str)
                
#                 results[current_mention_id] = llm_answer
            
#             # Handle potential multi-line answers
#             elif current_mention_id and ('"MESH:' in line or ']' in line):
#                 additional_items = re.findall(r'"([^"]*)"', line)
#                 results[current_mention_id].extend(additional_items)

#     return results




In [76]:
# log_file_path = "/home/cye73/llm_disambiguator/experiments/llm7.log"
# parsed_results = parse_log_file(log_file_path)
# parsed_results
# len(parsed_results)


In [79]:
# path = f"data/nlmchem/sapbert/recall/gpt-4o-mini_k=3_cands=10_recall=True5_true_results.json"
# with open(path, "w") as f:
#     json.dump(parsed_results, f, indent=4)

# RECALL

In [37]:
with open(
    "data/nlmchem/crossencoder/recall/Qwen2.5-7B-Instruct_k=3_cands=10_recall=True5_true_results.json", "r"
) as f:
    results = json.load(f)
len(mention2gold)

10025

In [38]:
recall_r, errors = recall_fn(results=results, mention2gold=mention2gold, ks = list(range(1,6)))
print(recall_r)
print("Number of mentions being ignored because not formatted the correct way by the LLM:", errors)

Error decoding for mention_id 3344250.34: unexpected EOF while parsing (<unknown>, line 0)
Error decoding for mention_id 3344250.35: unexpected EOF while parsing (<unknown>, line 0)
Error decoding for mention_id 3557226.239: unexpected EOF while parsing (<unknown>, line 0)
Error decoding for mention_id 3557226.257: unexpected EOF while parsing (<unknown>, line 0)
Error decoding for mention_id 3557226.279: unexpected EOF while parsing (<unknown>, line 0)
Error decoding for mention_id 3557226.282: unexpected EOF while parsing (<unknown>, line 0)
Error decoding for mention_id 3557226.295: unexpected EOF while parsing (<unknown>, line 0)
Error decoding for mention_id 3720569.67: unexpected EOF while parsing (<unknown>, line 0)
Error decoding for mention_id 3720569.72: unexpected EOF while parsing (<unknown>, line 0)
Error decoding for mention_id 3720569.74: unexpected EOF while parsing (<unknown>, line 0)
Error decoding for mention_id 3720569.76: unexpected EOF while parsing (<unknown>, li

In [39]:
cands = 10 # MODIFY THIS PARAMETER FOR TESTING !
number_hits = {"biencoder": number_hits_biencoder,
               "crossencoder": number_hits_crossencoder,
               "sapbert": number_hits_sapbert}
number_cands = sum([number_hits[f"{EL_model}"][i] for i in range(1, cands+1)])
unnormalized_recall = {
    f"normalized_{k}": v * number_cands / total_mentions
    for k, v in recall_r.items()
}
unnormalized_recall


{'normalized_recall@1': 0.8076244823784426,
 'normalized_recall@2': 0.8390265793621409,
 'normalized_recall@3': 0.846084848867651,
 'normalized_recall@4': 0.8486200640369772,
 'normalized_recall@5': 0.8494411280406792}

# ACCURACY

In [29]:
with open(
    "data/nlmchem/crossencoder/accuracy/Mistral-Nemo-Instruct-2407_k=3_cands=10_recall=False5_true_results.json", "r"
) as f:
    results = json.load(f)


In [30]:
cands = 10 # MODIFY THIS PARAMETER FOR TESTING !
number_hits = {"biencoder": number_hits_biencoder,
               "crossencoder": number_hits_crossencoder,
               "sapbert": number_hits_sapbert}
score, nb_cands, errors = scoring(results = results, mention2gold = mention2gold)
print("Number of mentions:", total_mentions)
# Number of candidates - the number of mentions where the answer is 'None' because of token limits imposed by the LLM
print("Number of evaluated candidates:", nb_cands)
print("Number of mentions being ignored because not correctly formatted by the LLM:", errors)
print("Accuracy :", score)
print("Unormalized accuracy:", 
      score * nb_cands / total_mentions)

Number of mentions: 11624
Number of evaluated candidates: 10002
Number of mentions being ignored because not correctly formatted by the LLM: 0
Accuracy : 0.9359128174365127
Unormalized accuracy: 0.8053165863730213


# PLOTS

In [ ]:
# Run time vs dataset size
dataset_size = [860]

llama_run_time = [14]
mistral_run_time = [20]
qwen_run_time = [11]
gpt4o-mini_run_time = [11]
gpt4o_run_time = [12]


# vllm + aqlm


In [ ]:
llm = LLM(
    model="ISTA-DASLab/Mixtral-8x7B-Instruct-v0_1-AQLM-2Bit-1x16-hf",
    enforce_eager=True,
)

tokenizer = llm.get_tokenizer()

In [30]:
# def prompt_vllm_aqlm(mention, context, system_instructions, candidates, topk_examples, llm, tokenizer, sampling_params):
#     ''' 
#     mention : str (name of the mention to be linked)
#     context : str (context where the mention appears)
#     system_instructions : str (instructions for the LLM)
#     candidates : list of list of CUIs : [[cui1], [cui2, cui3], ...]
#     topk_examples : str (top k examples of similar contexts)
#     llm : LLM model
#     tokenizer : AutoTokenizer
#     sampling_params : SamplingParams config
#     '''
    
#     prompt_text = f"""
#     System Instructions: {system_instructions} \n
    
#     Here are a few examples: \n
#     {topk_examples} \n

#     This is the specific mention that needs to be linked to the correct entity: {mention} \n

#     This is the context where the mention appears: \n
#     {context} \n
    
#     These are the candidate entities to choose from: \n
#     {candidates} \n
    
#     You MUST PROVIDE an ANSWER among the candidates. \n

#     Return the answer in the following format: CUI
#     For instance : "MESH:D000000" "OMIM:000000" are valid answers. \n
#     Do not add provide any explanations ! But you MUST give ONE answer.
#     """
#     conversations = tokenizer.apply_chat_template(
#         [{'role': 'user', 'content': prompt_text}],
#         tokenize=False,
#     )

#     # Decode the generated tokens into text
#     outputs = llm.generate([conversations], sampling_params=sampling_params, use_tqdm=False)
#     answer = outputs[0].outputs[0].text

#     return answer


In [ ]:
mention = "9931324.1"
text = mention2text[mention]
context = mention2context[mention]
candidates = get_candidates_data(mention2biencoder_candidates[mention], ontology2)
topk = topk_examples(model = model,
                    index = index,
                    query=context, 
                    corpus=corpus, 
                    TrainMap_context2mention=TrainMap_context2mention, 
                    train_mention2text=train_mention2text, 
                    train_mention2gold=train_mention2gold, 
                    ontology=ontology2, 
                    k=3)

pred_cui = prompt_vllm_aqlm(mention=text, 
                       context=context, 
                       system_instructions=system_instructions,
                       candidates=candidates, 
                       topk_examples=topk,
                       llm=llm,
                       tokenizer=tokenizer,
                       sampling_params=sampling_params)

# Output the results
print("Mention ID:", mention)
print("Context:", context)
print("Top k examples:", topk)
print("Text:", mention2text[mention])
print("Gold CUI:", mention2gold[mention])
print("Predicted CUI:", pred_cui)

In [44]:
# def evaluate_vllm_aqlm(
#     llm,
#     nlp_model,
#     tokenizer,
#     index,
#     system_instructions,
#     mentions,
#     ontology,
#     corpus,
#     mention2context,
#     mention2biencoder_candidates,
#     mention2text,
#     TrainMap_context2mention,
#     train_mention2text,
#     train_mention2gold,
#     k,
#     sampling_params,
# ):
#     """
#     Run "prompt" function for each mention in the list of mentions.
#     Returns a dictionary {mention_id : predicted CUI}
#     -------
#     llm : LLM model
#     nlp_model : SentenceTransformer model
#     tokenizer : AutoTokenizer
#     index : faiss index
#     system_instructions : str (instructions for the LLM)
#     mentions : list (mention_ids)
#     ontology : BiomedicalOntology object
#     corpus : list of str (all context sentences)
#     mention2context : dict (mention_id : context)
#     mention2biencoder_candidates : dict (mention_id : list of candidate CUIs)
#     mention2text : dict (mention_id : mention name)
#     TrainMap_context2mention : dict (context sentence to mention_id)
#     train_mention2text : dict (mention_id to mention name)
#     train_mention2gold : dict (mention_id to gold CUI)
#     k : int (number of nearest neighbors)
#     sampling_params : SamplingParams config
#     """
#     results = {}
#     for i in range(len(mentions)) :
#         mention_id = mentions[i]
#         mention_name = mention2text[mention_id]
#         context = mention2context[mention_id]
#         candidates = get_candidates_data(mention2biencoder_candidates[mentions[i]], ontology)
#         # candidates = get_candidates_data_v2(mention2crossencoder_candidates[mention])
#         topk = topk_examples(
#             model=nlp_model,  # sentence transformer model
#             index=index,
#             query=context,
#             corpus=corpus,
#             TrainMap_context2mention=TrainMap_context2mention,
#             train_mention2text=train_mention2text,
#             train_mention2gold=train_mention2gold,
#             ontology=ontology,
#             k=k,
#         )
#         text = prompt_vllm_aqlm(
#             mention=mention_name,
#             context=context,
#             system_instructions=system_instructions,
#             candidates=candidates,
#             topk_examples=topk,
#             llm=llm,
#             tokenizer=tokenizer,
#             sampling_params=sampling_params,
#         )
#         print("mention ID :", mention_id , "|| LLM answer :", text)
#         cand = extract_cui(text)
#         results[mention_id] = cand
#         if i%20 == 0:
#             print(f"i = {i}")

#     return results
    


In [ ]:
results = evaluate_vllm_aqlm(llm=llm,
                        nlp_model=model,
                        tokenizer=tokenizer,
                        index=index,
                        system_instructions=system_instructions,
                        mentions=mentions[:5],
                        ontology=ontology2,
                        corpus=corpus,
                        mention2context=mention2context,
                        mention2biencoder_candidates=mention2biencoder_candidates,
                        mention2text=mention2text,
                        TrainMap_context2mention=TrainMap_context2mention,
                        train_mention2text=train_mention2text,
                        train_mention2gold=train_mention2gold,
                        k=10,
                        sampling_params=sampling_params)
results

In [ ]:
results

In [ ]:
score = scoring(results=results, mention2gold=mention2gold)
score

### Fine-tuning

In [120]:
from datasets import load_metric

bleu = load_metric("bleu")

prediction1 = [["there", "is", "a", "need", "for", "adequate", "and", "predictable", "resources"]]
prediction2 = [["resources", "be", "sufficient", "and", "predictable", "to"]] 

reference1 = [[["resources", "have", "to", "be", "sufficient", "and", "they", "have", "to", "be", "predictable"]]]
reference2 = [[["adequate", "and", "predictable", "resources", "are", "required"]]]

bleu_c1r1 = bleu.compute(predictions=prediction1, references=reference1, max_order=2)
print("bleu score for c1 using r1 :", bleu_c1r1)
bleu_c1r2 = bleu.compute(predictions=prediction1, references=reference2, max_order=2)
print("bleu score for c1 using r2 :", bleu_c1r2)

bleu_c2r1 = bleu.compute(predictions=prediction2, references=reference1, max_order=2)
print("bleu score for c2 using r1 :", bleu_c2r1)
bleu_c2r2 = bleu.compute(predictions=prediction2, references=reference2, max_order=2)
print("bleu score for c2 using r2 :", bleu_c2r2)

/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/datasets/load.py:756: FutureWarning: The repository for bleu contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/bleu/bleu.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


bleu score for c1 using r1 : {'bleu': 0.0, 'precisions': [0.3333333333333333, 0.0], 'brevity_penalty': 0.8007374029168082, 'length_ratio': 0.8181818181818182, 'translation_length': 9, 'reference_length': 11}
bleu score for c1 using r2 : {'bleu': 0.408248290463863, 'precisions': [0.4444444444444444, 0.375], 'brevity_penalty': 1.0, 'length_ratio': 1.5, 'translation_length': 9, 'reference_length': 6}
bleu score for c2 using r1 : {'bleu': 0.2748640411822265, 'precisions': [1.0, 0.4], 'brevity_penalty': 0.43459820850707814, 'length_ratio': 0.5454545454545454, 'translation_length': 6, 'reference_length': 11}
bleu score for c2 using r2 : {'bleu': 0.316227766016838, 'precisions': [0.5, 0.2], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 6, 'reference_length': 6}


In [ ]:
from datasets import load_metric

bleu = load_metric("bleu")

prediction1 = ["there", "is", "a", "need", "for", "adequate", "and", "predictable", "resources"]
prediction2 = ["resources", "be", "sufficient", "and", "predictable", "to"]

reference1 = [["resources", "have", "to", "be", "sufficient", "and", "they", "have", "to", "be", "predictable"]]
reference2 = [["adequate", "and", "predictable", "resources", "are", "required"]]

bleu_c1r1 = bleu.compute(predictions=[prediction1], references=reference1, max_order=2)
print("bleu score for c1 using r1 :", bleu_c1r1)

bleu_c1r2 = bleu.compute(predictions=[prediction1], references=reference2, max_order=2)
print("bleu score for c1 using r2 :", bleu_c1r2)

bleu_c2r1 = bleu.compute(predictions=[prediction2], references=reference1, max_order=2)
print("bleu score for c2 using r1 :", bleu_c2r1)

bleu_c2r2 = bleu.compute(predictions=[prediction2], references=reference2, max_order=2)
print("bleu score for c2 using r2 :", bleu_c2r2)

In [ ]:
# from transformers import AutoModel, AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained('ucaslcl/GOT-OCR2_0', trust_remote_code=True)
# # Load the model directly onto device 0
# model = AutoModel.from_pretrained(
#     'ucaslcl/GOT-OCR2_0',
#     trust_remote_code=True,
#     low_cpu_mem_usage=True,
#     use_safetensors=True,
#     pad_token_id=tokenizer.eos_token_id,
#     device_map="cuda:0"  # Explicitly map the model to device 0
# )

# model = model.eval()

# # input your test image
# image_file = 'OCR_test.jpg'

# # plain texts OCR
# res = model.chat(tokenizer, image_file, ocr_type='ocr')

# # format texts OCR:
# # res = model.chat(tokenizer, image_file, ocr_type='format')

# # fine-grained OCR:
# # res = model.chat(tokenizer, image_file, ocr_type='ocr', ocr_box='')
# # res = model.chat(tokenizer, image_file, ocr_type='format', ocr_box='')
# # res = model.chat(tokenizer, image_file, ocr_type='ocr', ocr_color='')
# # res = model.chat(tokenizer, image_file, ocr_type='format', ocr_color='')

# # multi-crop OCR:
# # res = model.chat_crop(tokenizer, image_file, ocr_type='ocr')
# # res = model.chat_crop(tokenizer, image_file, ocr_type='format')

# # render the formatted OCR results:
# # res = model.chat(tokenizer, image_file, ocr_type='format', render=True, save_render_file = './demo.html')

# print(res)
